# Artificial Intelligence – 2024/2025
## Final Project – Group: [YOUR GROUP]

**Course:** Artificial Intelligence (IA)  
**Academic Year:** 2024/2025  
**Project Title:** Adversarial Search and Decision Trees in PopOut

**Group Members:**
- [Name] – [Student ID]
- [Name] – [Student ID]
- [Name] – [Student ID]

---

# AI Project: PopOut, MCTS, and ID3 Decision Trees

This notebook presents the final project for the *Artificial Intelligence* course. The project is structured into **three main parts**, combining search-based AI and supervised machine learning techniques applied to the game of **PopOut** — a variant of Connect Four:

- **Part 1 – PopOut Game and Monte Carlo Tree Search (MCTS):**  
  We implemented the game *PopOut* in Python, including the full game logic and terminal interface. An intelligent agent was developed using **Monte Carlo Tree Search (MCTS)**, enhanced with the **UCT formula** (Upper Confidence Bound for Trees), **domain-specific heuristics** for simulation guidance, and a **discount factor** in backpropagation. The agent supports three difficulty levels: Easy, Medium, and Hard.

- **Part 2 – ID3 Decision Tree Agents:**  
  This part explores two distinct uses of the **ID3 algorithm**:
  1. A **learning-based agent** trained on gameplay data generated by MCTS self-play. The tree learns to imitate MCTS decisions on PopOut board states.
  2. The same ID3 implementation applied to the classical **Iris dataset**, demonstrating the algorithm's generalisability to standard classification tasks with continuous attributes via **dynamic discretisation**.

- **Part 3 – MCTS Performance Analysis:**  
  A tournament framework simulates multiple games between MCTS agents of different difficulty levels and produces visualisations covering win rates, computational cost per move, and move-type distributions (drop vs. pop).

The notebook includes game demonstrations, training and evaluation of decision trees, performance comparisons, and detailed explanations of each algorithmic component.

---

# Part 1 – PopOut Game and Monte Carlo Tree Search (MCTS)

This section covers the implementation of the **PopOut** game and the design of an intelligent agent based on **MCTS**. We include the full game logic, heuristic-guided simulation, and the complete MCTS algorithm.

## 1.1 Imports and Initialization

We begin by importing all libraries required throughout the project.

In [1]:
import math
import random
import time
import copy
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict, Counter

plt.style.use('ggplot')
sns.set_palette("muted")

## 1.2 PopOut Game Rules and Logic

**PopOut** is a variant of *Connect Four* played on a 6×7 grid. The goal is to align **four tokens** horizontally, vertically, or diagonally. Unlike standard Connect Four, each player may also use a **pop** move:

- **Drop (`drop`):** Place your token into any non-full column. The piece falls to the lowest available row.
- **Pop (`pop`):** Remove your own token from the **bottom** of any column where your token currently sits there. All pieces in that column slide down by one row.

### Special rules:
- A player may only pop their **own** token from the bottom.
- If both players simultaneously achieve four in a row (which can happen after a pop), the **player who moved** wins.
- The game is a draw if the board is full with no valid moves, or if the same board state repeats **three times** (threefold-repetition draw).

### Key methods in `PopOut`:
- `get_valid_moves()` – returns all legal `(move_type, column)` pairs for the current player.
- `make_move(move_type, col)` – applies the move and returns the winner (0 = no winner yet, -1 = draw, 1 or 2 = winning player).
- `check_winner(last_player)` – detects four-in-a-row for both players, handling simultaneous wins.
- `clone()` – returns a deep copy of the game state for simulations.

In [2]:
class PopOut:
    def __init__(self):
        self.ROWS = 6
        self.COLS = 7
        # 0 = Empty, 1 = X (Player 1), 2 = O (Player 2)
        self.board = [[0 for _ in range(self.COLS)] for _ in range(self.ROWS)]
        self.current_player = 1
        self.state_history = {}
        self._record_state()

    def print_board(self):
        symbols = {0: " . ", 1: " X ", 2: " O "}
        header = "  " + "   ".join([str(i) for i in range(self.COLS)])
        print("\n" + header)
        print(" \u250c" + "\u2500\u2500\u2500\u252c" * (self.COLS - 1) + "\u2500\u2500\u2500\u2510")
        for i, row in enumerate(self.board):
            row_str = " \u2502" + "\u2502".join([symbols[cell] for cell in row]) + "\u2502"
            print(row_str)
            if i < self.ROWS - 1:
                print(" \u251c" + "\u2500\u2500\u2500\u253c" * (self.COLS - 1) + "\u2500\u2500\u2500\u2524")
        print(" \u2514" + "\u2500\u2500\u2500\u2534" * (self.COLS - 1) + "\u2500\u2500\u2500\u2518")
        print(f" Turn: Player {'X' if self.current_player == 1 else 'O'}")

    def get_valid_moves(self):
        moves = []
        for col in range(self.COLS):
            if self.board[0][col] == 0:
                moves.append(('drop', col))
            if self.board[self.ROWS - 1][col] == self.current_player:
                moves.append(('pop', col))
        if self.is_board_full():
            moves.append(('draw_declaration', 'board_full'))
        return moves

    def make_move(self, move_type, col):
        if move_type == 'draw_declaration':
            return -1
        last_player = self.current_player
        if move_type == 'drop':
            for r in range(self.ROWS - 1, -1, -1):
                if self.board[r][col] == 0:
                    self.board[r][col] = self.current_player
                    break
        elif move_type == 'pop':
            for r in range(self.ROWS - 1, 0, -1):
                self.board[r][col] = self.board[r - 1][col]
            self.board[0][col] = 0
        winner = self.check_winner(last_player)
        self.current_player = 3 - self.current_player
        self._record_state()
        return winner

    def _record_state(self):
        state_key = (tuple(tuple(row) for row in self.board), self.current_player)
        self.state_history[state_key] = self.state_history.get(state_key, 0) + 1

    def is_repetition_draw(self):
        state_key = (tuple(tuple(row) for row in self.board), self.current_player)
        return self.state_history.get(state_key, 0) >= 3

    def is_board_full(self):
        return all(self.board[0][col] != 0 for col in range(self.COLS))

    def check_winner(self, last_player):
        p1 = self._has_four(1)
        p2 = self._has_four(2)
        if p1 and p2: return last_player
        if p1: return 1
        if p2: return 2
        return 0

    def _has_four(self, p):
        for r in range(self.ROWS):
            for c in range(self.COLS - 3):
                if all(self.board[r][c+i] == p for i in range(4)): return True
        for r in range(self.ROWS - 3):
            for c in range(self.COLS):
                if all(self.board[r+i][c] == p for i in range(4)): return True
        for r in range(self.ROWS - 3):
            for c in range(self.COLS - 3):
                if all(self.board[r+i][c+i] == p for i in range(4)): return True
        for r in range(3, self.ROWS):
            for c in range(self.COLS - 3):
                if all(self.board[r-i][c+i] == p for i in range(4)): return True
        return False

    def clone(self):
        new_game = PopOut()
        new_game.board = [row[:] for row in self.board]
        new_game.current_player = self.current_player
        new_game.state_history = self.state_history.copy()
        return new_game

### Quick Game State Demo

We can visualise a fresh board and test the move system:

In [3]:
demo = PopOut()
print("Initial board:")
demo.print_board()
print("\nValid moves for Player 1 (first few):", demo.get_valid_moves()[:5], "...")

# Drop in center, then show board
demo.make_move('drop', 3)
demo.make_move('drop', 3)
print("\nAfter two drops in column 3:")
demo.print_board()

Initial board:

  0   1   2   3   4   5   6
 ┌───┬───┬───┬───┬───┬───┬───┐
 │ . │ . │ . │ . │ . │ . │ . │
 ├───┼───┼───┼───┼───┼───┼───┤
 │ . │ . │ . │ . │ . │ . │ . │
 ├───┼───┼───┼───┼───┼───┼───┤
 │ . │ . │ . │ . │ . │ . │ . │
 ├───┼───┼───┼───┼───┼───┼───┤
 │ . │ . │ . │ . │ . │ . │ . │
 ├───┼───┼───┼───┼───┼───┼───┤
 │ . │ . │ . │ . │ . │ . │ . │
 ├───┼───┼───┼───┼───┼───┼───┤
 │ . │ . │ . │ . │ . │ . │ . │
 └───┴───┴───┴───┴───┴───┴───┘
 Turn: Player X

Valid moves for Player 1 (first few): [('drop', 0), ('drop', 1), ('drop', 2), ('drop', 3), ('drop', 4)] ...

After two drops in column 3:

  0   1   2   3   4   5   6
 ┌───┬───┬───┬───┬───┬───┬───┐
 │ . │ . │ . │ . │ . │ . │ . │
 ├───┼───┼───┼───┼───┼───┼───┤
 │ . │ . │ . │ . │ . │ . │ . │
 ├───┼───┼───┼───┼───┼───┼───┤
 │ . │ . │ . │ . │ . │ . │ . │
 ├───┼───┼───┼───┼───┼───┼───┤
 │ . │ . │ . │ . │ . │ . │ . │
 ├───┼───┼───┼───┼───┼───┼───┤
 │ . │ . │ . │ O │ . │ . │ . │
 ├───┼───┼───┼───┼───┼───┼───┤
 │ . │ . │ . │ X │ . │ . │ .

---

## 1.3 Monte Carlo Tree Search (MCTS) – Theory

**Monte Carlo Tree Search (MCTS)** is a decision-making algorithm that builds a search tree incrementally by simulating random (or heuristic-guided) games and propagating results back through the tree. It does not require a hand-crafted evaluation function.

### The Four Phases

Each iteration of MCTS consists of four phases:

1. **Selection** – Starting from the root, repeatedly choose the child with the highest **UCT score** until reaching a node with unexplored children or a terminal state.

2. **Expansion** – Add one new child node by applying an unexplored move to the current state.

3. **Simulation (Rollout/Playout)** – Play out the game from the new node using a simulation policy (random or heuristic-guided) until reaching a terminal state.

4. **Backpropagation** – Propagate the simulation result back up through all ancestor nodes, updating visit counts and win statistics.

### UCT Formula

To select which child to visit during the **Selection** phase, we use the **UCT (Upper Confidence Bound for Trees)** formula:

$$UCT(v, c) = \frac{w_v}{n_v} + C \cdot \sqrt{\frac{\ln N_{parent}}{n_v}}$$

Where:
- $w_v$ = accumulated wins (with discount) for node $v$
- $n_v$ = visit count for node $v$
- $N_{parent}$ = visit count of the parent node
- $C$ = exploration constant (typically $\sqrt{2} \approx 1.414$)

The **first term** exploits known good moves; the **second term** encourages exploration of less-visited nodes.

### Enhancements in Our Implementation

Beyond the standard MCTS, we implemented several improvements:

| Enhancement | Description |
|---|---|
| **Heuristic simulation** | Three simulation policies: `random`, `light` (centre-bias), `heavy` (adjacency-aware) |
| **Discount factor (γ = 0.95)** | Backpropagation rewards faster wins more than slow wins, and penalises slow defeats more than fast ones |
| **Tactical vision (1-step lookahead)** | Before running MCTS, check for immediate winning moves (DM) or forced blocks (ADM) |
| **Dynamic C parameter** | Player 1 uses a lower C (exploitation-focused); Player 2 uses a higher C (exploration-focused) |
| **Probabilistic selection (Easy mode)** | Easy mode samples from the top-3 children instead of always choosing the best |
| **Time limit (Hard mode)** | Hard mode runs for up to 4.5 seconds regardless of iteration count |

## 1.4 Heuristic Simulation Policy

Instead of choosing random moves during rollouts, we use domain knowledge to guide the simulation:

- **`random`**: Chooses any legal move uniformly at random.
- **`light`**: Prefers central columns and pieces adjacent to existing pieces (mild bias).
- **`heavy`**: Strongly prefers extending own sequences and blocking opponent sequences — used in Hard mode.

Pop moves are given lower weight since they are less frequently beneficial at early stages.

In [4]:
def heuristic_choice(game, moves, heur_type):
    """Heuristic simulation policy to guide MCTS rollouts."""
    if heur_type == 'random':
        return random.choice(moves)

    board      = game.board
    rows       = game.ROWS
    cols_count = game.COLS
    center     = cols_count // 2
    current    = game.current_player
    opponent_p = 3 - current

    weights = []
    for m_type, col in moves:
        if m_type == 'drop':
            dist_to_center = abs(col - center)
            w = max(1, 10 - dist_to_center * 2)
            if heur_type in ('light', 'heavy'):
                landing = next((r for r in range(rows - 1, -1, -1) if board[r][col] == 0), None)
                if landing is not None:
                    own_adj = sum(1 for dc in (-1, 1) if 0 <= col + dc < cols_count and board[landing][col + dc] == current)
                    opp_adj = sum(1 for dc in (-1, 1) if 0 <= col + dc < cols_count and board[landing][col + dc] == opponent_p)
                    if heur_type == 'heavy':
                        w += own_adj * 15
                        w += opp_adj * 12
                    else:
                        w += own_adj * 5
                        w += opp_adj * 4
            weights.append(w)
        elif m_type == 'pop':
            weights.append(1 if heur_type == 'heavy' else 2)
        else:
            weights.append(1)

    return random.choices(moves, weights=weights, k=1)[0]

## 1.5 MCTS Node and Tree Structure

Each node in the MCTS tree represents a game state. It stores:
- The game state (`state`)
- Its parent node and the move that led to it
- A list of child nodes (`children`)
- Visit count (`visits`) and accumulated results (`results`)
- Untried moves not yet expanded

The `results` dictionary uses player IDs as keys and float values (due to the discount factor). A draw is stored with key `-1` and value `0.5`.

In [5]:
class MCTSNode:
    """Node in the MCTS search tree."""
    def __init__(self, state, parent=None, move=None, heur_type='light'):
        self.state = state
        self.parent = parent
        self.move = move
        self.children = []
        self.visits = 0
        self.results = defaultdict(float)
        self.untried_moves = self.state.get_valid_moves()
        random.shuffle(self.untried_moves)  # reduce left-to-right bias
        self.heur_type = heur_type

    def is_fully_expanded(self):
        return len(self.untried_moves) == 0

    def best_child(self, c_param=1.414):
        """Selection phase: UCT formula."""
        player_to_optimize = self.state.current_player
        choices_weights = []
        for child in self.children:
            if child.visits == 0:
                choices_weights.append(0)
                continue
            win_rate  = (child.results[player_to_optimize] + child.results[-1]) / child.visits
            exploration = c_param * math.sqrt(math.log(self.visits) / child.visits)
            choices_weights.append(win_rate + exploration)
        return self.children[choices_weights.index(max(choices_weights))]

    def expand(self):
        """Expansion phase: create one new child node."""
        move = self.untried_moves.pop(0)
        next_state = self.state.clone()
        next_state.make_move(*move)
        child_node = MCTSNode(next_state, parent=self, move=move, heur_type=self.heur_type)
        self.children.append(child_node)
        return child_node

    def simulate(self):
        """Simulation phase: play to terminal using heuristic policy."""
        temp_state = self.state.clone()
        depth = 0
        while True:
            valid_moves = temp_state.get_valid_moves()
            if not valid_moves:
                return -1, depth  # draw
            move = heuristic_choice(temp_state, valid_moves, self.heur_type)
            res  = temp_state.make_move(*move)
            depth += 1
            if res != 0:
                return res, depth

    def backpropagate(self, result, depth):
        """Backpropagation phase with discount factor γ = 0.95."""
        self.visits += 1
        if result == -1:
            self.results[-1] += 0.5
        else:
            discount = 0.95 ** depth
            self.results[result] += discount
        if self.parent:
            self.parent.backpropagate(result, depth + 1)

## 1.6 MCTS Move Selection Function

The main entry point for the MCTS agent. It selects a move by:
1. Applying **tactical vision** (check immediate wins/losses before any simulation).
2. Running the MCTS loop for a configured number of iterations (or time budget for Hard).
3. Returning the move of the **most-visited** child (or sampling from top-3 for Easy).

### Difficulty Settings

| Level | Iterations | Time limit | Simulation | C parameter | Selection |
|---|---|---|---|---|---|
| Easy | 150 | None | Random | 2.5 (fixed) | Sample top-3 |
| Medium | 1000 | None | Light | 1.2 / 1.6 (dynamic) | Most visited |
| Hard | 10 000 | 4.5 s | Heavy | 0.96 / 1.28 (dynamic) | Most visited |

In [6]:
def get_mcts_move(game_state, dificuldade='medio'):
    valid_moves = game_state.get_valid_moves()

    # Dynamic C: P1 exploits (smaller C), P2 explores (larger C)
    is_player_one = (game_state.current_player == 1)
    base_c_param  = 1.2 if is_player_one else 1.6

    time_limit = None
    if dificuldade == 'facil':
        iterations         = 150
        c_param            = 2.5
        use_tactical_vision = True
        heur_type          = 'random'
    elif dificuldade == 'dificil':
        iterations         = 10000
        time_limit         = 4.5
        c_param            = base_c_param * 0.8
        use_tactical_vision = True
        heur_type          = 'heavy'
    else:  # medio
        iterations         = 1000
        c_param            = base_c_param
        use_tactical_vision = True
        heur_type          = 'light'

    # --- Tactical Vision: 1-step lookahead ---
    if use_tactical_vision:
        # Decisive Move: can we win immediately?
        for move in valid_moves:
            test_state = game_state.clone()
            if test_state.make_move(*move) == game_state.current_player:
                return move
        # Anti-Decisive Move: can opponent win on their next turn?
        opponent  = 3 - game_state.current_player
        opp_state = game_state.clone()
        opp_state.current_player = opponent
        opp_moves = opp_state.get_valid_moves()
        winning_threats = []
        for o_move in opp_moves:
            test_win = opp_state.clone()
            if test_win.make_move(*o_move) == opponent:
                winning_threats.append(o_move)
        blockable = [m for m in winning_threats if m[0] == 'drop' and ('drop', m[1]) in valid_moves]
        if len(winning_threats) == 1 and len(blockable) == 1:
            return blockable[0]

    # --- Main MCTS Loop ---
    root     = MCTSNode(game_state.clone(), heur_type=heur_type)
    deadline = time.time() + time_limit if time_limit is not None else None

    for i in range(iterations):
        if deadline is not None and time.time() >= deadline:
            break
        node = root
        # 1. Selection
        while not node.untried_moves and node.children:
            node = node.best_child(c_param=c_param)
        # 2. Expansion
        if node.untried_moves:
            node = node.expand()
        # 3. Simulation
        result, playout_depth = node.simulate()
        # 4. Backpropagation
        node.backpropagate(result, playout_depth)

    # --- Final Move Selection ---
    if dificuldade == 'facil':
        sorted_children = sorted(root.children, key=lambda c: c.visits, reverse=True)
        top_children    = sorted_children[:min(3, len(sorted_children))]
        return random.choice(top_children).move
    else:
        return max(root.children, key=lambda c: c.visits).move

### MCTS Quick Demo

We run the Medium MCTS agent for one move and observe its choice:

In [7]:
demo_game = PopOut()
demo_game.print_board()
print("\nRunning MCTS (Medium) for Player 1...")
chosen_move = get_mcts_move(demo_game, dificuldade='medio')
print(f"MCTS chose: {chosen_move}")


  0   1   2   3   4   5   6
 ┌───┬───┬───┬───┬───┬───┬───┐
 │ . │ . │ . │ . │ . │ . │ . │
 ├───┼───┼───┼───┼───┼───┼───┤
 │ . │ . │ . │ . │ . │ . │ . │
 ├───┼───┼───┼───┼───┼───┼───┤
 │ . │ . │ . │ . │ . │ . │ . │
 ├───┼───┼───┼───┼───┼───┼───┤
 │ . │ . │ . │ . │ . │ . │ . │
 ├───┼───┼───┼───┼───┼───┼───┤
 │ . │ . │ . │ . │ . │ . │ . │
 ├───┼───┼───┼───┼───┼───┼───┤
 │ . │ . │ . │ . │ . │ . │ . │
 └───┴───┴───┴───┴───┴───┴───┘
 Turn: Player X

Running MCTS (Medium) for Player 1...
MCTS chose: ('drop', 3)


---

# Part 2 – ID3 Decision Tree

This part covers two distinct applications of our custom **ID3 algorithm**:
1. Training a decision tree to **play PopOut**, using gameplay data generated by MCTS self-play.
2. Applying the same ID3 implementation to the **Iris dataset**, a classical classification benchmark requiring discretisation of continuous attributes.

## 2.1 ID3 Algorithm – Theory

**ID3 (Iterative Dichotomiser 3)**, introduced by Ross Quinlan in 1986, builds a decision tree by recursively selecting the attribute that best classifies the training data.

### Entropy

Entropy measures the **impurity** of a label distribution $S$:

$$H(S) = -\sum_{c \in C} p_c \log_2 p_c$$

where $p_c$ is the proportion of class $c$ in $S$. $H(S) = 0$ means all examples belong to the same class (pure leaf).

### Information Gain

Information Gain measures how much an attribute $A$ reduces entropy:

$$IG(S, A) = H(S) - \sum_{v \in V(A)} \frac{|S_v|}{|S|} H(S_v)$$

ID3 chooses the attribute with the **highest information gain** at each node.

### Recursive Construction

The tree is built recursively. A node becomes a **leaf** when:
- All examples at that node share the same label.
- No attributes remain to split on.
- A configured maximum depth is reached.

### Advantages and Limitations

| Advantage | Limitation |
|---|---|
| Fully interpretable | Overfits on small/noisy datasets |
| No hyperparameter tuning beyond depth | Cannot handle continuous values natively |
| Fast inference | Sensitive to attribute ordering |
| Works on categorical data directly | Does not support missing values |

## 2.2 ID3 Implementation

Our implementation uses a recursive `TreeNode` structure. The `ID3DecisionTree` class exposes:
- `fit(data, target_name, features, depth)` – trains the tree recursively.
- `predict_single(node, sample)` – traverses the tree for one example.
- `predict(data)` – applies prediction to a full DataFrame.
- `print_tree()` – displays the tree structure in the terminal.

In [8]:
class TreeNode:
    def __init__(self, attribute=None, label=None):
        self.attribute = attribute  # Attribute used for splitting
        self.label     = label      # If leaf: the predicted class
        self.children  = {}         # {attribute_value: TreeNode}


class ID3DecisionTree:
    def __init__(self, max_depth=None):
        self.root      = None
        self.max_depth = max_depth

    def _entropy(self, target_col):
        """Compute entropy H(S)."""
        counts = pd.Series(target_col).value_counts().values
        total  = sum(counts)
        return -sum((c / total) * math.log2(c / total) for c in counts)

    def _information_gain(self, data, split_attribute, target_name):
        """Compute Information Gain IG(S, A)."""
        total_entropy = self._entropy(data[target_name])
        vals, counts  = (pd.Series(data[split_attribute]).value_counts().index,
                         pd.Series(data[split_attribute]).value_counts().values)
        weighted_entropy = sum(
            (counts[i] / sum(counts)) * self._entropy(data[data[split_attribute] == vals[i]][target_name])
            for i in range(len(vals))
        )
        return total_entropy - weighted_entropy

    def fit(self, data, target_name, features, depth=0):
        """Build the decision tree recursively using ID3."""
        target_values = data[target_name].values
        # Base case 1: all labels are the same
        if len(set(target_values)) == 1:
            return TreeNode(label=target_values[0])
        # Base case 2: no attributes left or max depth reached
        if not features or (self.max_depth is not None and depth >= self.max_depth):
            return TreeNode(label=Counter(target_values).most_common(1)[0][0])
        # Choose best attribute by information gain
        gains       = [self._information_gain(data, f, target_name) for f in features]
        best_feature = features[gains.index(max(gains))]
        node         = TreeNode(attribute=best_feature)
        remaining    = [f for f in features if f != best_feature]
        for value in data[best_feature].unique():
            subset = data[data[best_feature] == value]
            if len(subset) == 0:
                node.children[value] = TreeNode(label=Counter(target_values).most_common(1)[0][0])
            else:
                node.children[value] = self.fit(subset, target_name, remaining, depth + 1)
        self.root = node
        return node

    def predict_single(self, node, sample):
        """Traverse the tree to classify one sample."""
        if node.label is not None:
            return node.label
        value = sample[node.attribute]
        if value in node.children:
            return self.predict_single(node.children[value], sample)
        return None  # unseen value

    def predict(self, data):
        """Classify all rows in a DataFrame."""
        return [self.predict_single(self.root, row) for _, row in data.iterrows()]

    def print_tree(self, node=None, indent=""):
        if node is None:
            node = self.root
        if node.label is not None:
            print(indent + "└── Class:", node.label)
            return
        print(indent + "├── Attribute:", node.attribute)
        for value, child_node in node.children.items():
            print(indent + "│   ├── Value (" + str(value) + "):")
            self.print_tree(child_node, indent + "│   │   ")

---

## 2.3 Dataset Generation via MCTS Self-Play (PopOut)

To train an ID3 tree that can play PopOut, we generate labelled data using MCTS self-play:

1. Two MCTS agents (Medium difficulty) play a complete game against each other.
2. At each turn, the **current board state** (42 cell values: 6 rows × 7 columns) is recorded as a feature vector.
3. The **move chosen by MCTS** is recorded as the label (`best_move`).
4. The resulting dataset is saved as `popout_dataset.csv`.

Board cells are encoded as:
- `0` – Empty
- `1` – Player 1 (X)
- `2` – Player 2 (O)

In [9]:
def generate_dataset(num_games=10, filename="popout_dataset.csv"):
    """Have MCTS agents self-play to build a training dataset."""
    print(f"Generating dataset from {num_games} MCTS self-play games...")
    dataset = []
    for i in range(num_games):
        game = PopOut()
        while True:
            # Flatten 6x7 board into 42 named feature columns
            state_row = {f'cell_{r}_{c}': game.board[r][c]
                         for r in range(game.ROWS) for c in range(game.COLS)}
            move = get_mcts_move(game, dificuldade='medio')
            state_row['best_move'] = f"{move[0]}_{move[1]}"
            dataset.append(state_row)
            res = game.make_move(*move)
            if res != 0 or not game.get_valid_moves():
                break
        print(f"Game {i+1}/{num_games} done. Total samples: {len(dataset)}")
    df = pd.DataFrame(dataset)
    df.to_csv(filename, index=False)
    print(f"Dataset saved as '{filename}'.")
    return df

### Run Dataset Generation

If `popout_dataset.csv` already exists, we load it directly. Otherwise, we generate it now (this may take a few minutes).

In [10]:
import os

CSV_PATH = "popout_dataset.csv"

if os.path.exists(CSV_PATH):
    print(f"Loading existing dataset from '{CSV_PATH}'...")
    df_popout = pd.read_csv(CSV_PATH)
    print(f"Loaded {len(df_popout)} samples.")
else:
    df_popout = generate_dataset(num_games=25, filename=CSV_PATH)

df_popout.head()

Loading existing dataset from 'popout_dataset.csv'...
Loaded 6558 samples.


,cell_0,cell_1,cell_2,cell_3,cell_4,cell_5,cell_6,cell_7,cell_8,cell_9,...,cell_34,cell_35,cell_36,cell_37,cell_38,cell_39,cell_40,cell_41,current_player,target_move
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,drop_3
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,2,drop_3
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,1,drop_4
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,1,0,0,2,drop_2
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,2,1,1,0,0,1,drop_4


## 2.4 Training and Evaluating the ID3 Tree on PopOut

We split the dataset into **85% training / 15% test**, train the ID3 tree with `max_depth=6`, and measure its accuracy in predicting MCTS moves.

Note: even a 20–40% accuracy is meaningful for a game like PopOut, because many moves are roughly equivalent in many board positions.

In [11]:
target_col   = 'best_move'
feature_cols = [col for col in df_popout.columns if col != target_col]

train_data = df_popout.sample(frac=0.85, random_state=42)
test_data  = df_popout.drop(train_data.index)

print(f"Training samples: {len(train_data)} | Test samples: {len(test_data)}")

tree_popout = ID3DecisionTree(max_depth=6)
tree_popout.fit(train_data, target_name=target_col, features=feature_cols)

predictions = tree_popout.predict(test_data)
correct     = sum(p == r for p, r in zip(predictions, test_data[target_col]))
accuracy    = correct / len(test_data)

print(f"\nID3 accuracy on test data: {correct}/{len(test_data)} ({accuracy:.2%})")
print("(20–40% is typical for a single ID3 tree on a complex game like PopOut.)")

Training samples: 5574 | Test samples: 984


KeyError: 'best_move'

### Move Distribution in the Dataset

We visualise which moves MCTS preferred most often, which helps explain the label space the ID3 tree must learn from.

In [ ]:
move_counts = df_popout['best_move'].value_counts().head(15)

plt.figure(figsize=(12, 5))
move_counts.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Top 15 Most Frequent MCTS Moves in Training Dataset', fontsize=13, fontweight='bold')
plt.xlabel('Move (type_column)')
plt.ylabel('Frequency')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Drop vs Pop split
df_popout['move_type'] = df_popout['best_move'].apply(lambda m: m.split('_')[0])
type_counts = df_popout['move_type'].value_counts()
print("\nMove type distribution:")
print(type_counts.to_string())
print(f"Pop rate: {type_counts.get('pop', 0) / len(df_popout):.1%}")

### ID3 Tree Player for PopOut

We wrap the trained tree in a player class that can be used in the game loop. If the tree predicts an illegal or unknown move, the player falls back to the first legal move.

In [ ]:
def get_id3_move(game, tree):
    """Ask the trained ID3 tree for the best move given the current board."""
    state_row = {f'cell_{r}_{c}': game.board[r][c]
                 for r in range(game.ROWS) for c in range(game.COLS)}
    predicted_str = tree.predict_single(tree.root, state_row)
    valid_moves   = game.get_valid_moves()
    if predicted_str:
        parts = predicted_str.split('_')
        try:
            move = (parts[0], int(parts[1]))
            if move in valid_moves:
                return move
        except (IndexError, ValueError):
            pass
    print("  (ID3 fallback: first legal move)")
    return valid_moves[0]

---

## 2.5 ID3 on the Iris Dataset

To validate that our ID3 implementation generalises beyond game data, we apply it to the **Iris dataset** — a canonical supervised learning benchmark.

The Iris dataset contains 150 samples of three flower species (*Iris setosa*, *Iris versicolor*, *Iris virginica*) described by four numerical features:
- `sepal_length`, `sepal_width`, `petal_length`, `petal_width`

Because ID3 requires **categorical** features, we apply **quantile-based discretisation**: each numerical attribute is split into three equal-frequency bins labelled `Low`, `Medium`, `High`.

In [ ]:
def discretize_data(df, columns):
    """Convert continuous columns into 3 equal-frequency categories."""
    df_disc = df.copy()
    for col in columns:
        df_disc[col] = pd.qcut(df[col], q=3, labels=["Low", "Medium", "High"])
    return df_disc

In [ ]:
# Load Iris dataset
iris_path = "iris_1.csv"
col_names = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'class']

if os.path.exists(iris_path):
    iris_df = pd.read_csv(iris_path, header=None, names=col_names)
    print(f"Iris dataset loaded from local file ({len(iris_df)} samples).")
else:
    iris_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
    iris_df  = pd.read_csv(iris_url, header=None, names=col_names)
    print(f"Iris dataset loaded from UCI ({len(iris_df)} samples).")

iris_df.head()

In [ ]:
features_iris = col_names[:-1]

# Discretise
iris_disc  = discretize_data(iris_df, features_iris)

# Train/test split 80/20
iris_train = iris_disc.sample(frac=0.8, random_state=42)
iris_test  = iris_disc.drop(iris_train.index)

print(f"Training: {len(iris_train)} | Test: {len(iris_test)}")

tree_iris = ID3DecisionTree(max_depth=4)
tree_iris.fit(iris_train, target_name='class', features=features_iris)

iris_preds  = tree_iris.predict(iris_test)
iris_correct = sum(p == r for p, r in zip(iris_preds, iris_test['class']))
iris_acc     = iris_correct / len(iris_test)

print(f"\nIris accuracy: {iris_correct}/{len(iris_test)} ({iris_acc:.2%})")

In [ ]:
print("\nIris Decision Tree structure:")
tree_iris.print_tree()

### Visualisation: Iris Feature Distributions After Discretisation

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Iris Feature Distributions After Discretisation', fontsize=13, fontweight='bold')

for ax, feat in zip(axes, features_iris):
    iris_disc[feat].value_counts().reindex(['Low', 'Medium', 'High']).plot(
        kind='bar', ax=ax, color=['#4da6ff', '#ff9f40', '#ff6b6b'], edgecolor='white')
    ax.set_title(feat.replace('_', ' ').title())
    ax.set_xlabel('Category')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

---

# Part 3 – MCTS Performance Analysis

To quantitatively evaluate our MCTS implementation, we ran a **tournament** between agents of different difficulty levels. The tournament measures:

1. **Win rates** – Which difficulty level wins most often?
2. **Computational cost** – How long does each difficulty level take per move?
3. **Move-type distribution** – Do harder agents use `pop` moves more strategically?

Matchups simulated:
- **Easy vs Medium**
- **Medium vs Hard**

## 3.1 Tournament Simulation

In [ ]:
def simulate_game(dif_p1, dif_p2):
    """Run one silent game between two MCTS bots and record turn-level metrics."""
    game      = PopOut()
    turn_data = []
    while True:
        current    = game.current_player
        difficulty = dif_p1 if current == 1 else dif_p2
        start      = time.time()
        move       = get_mcts_move(game, dificuldade=difficulty)
        elapsed    = time.time() - start
        turn_data.append({
            'Player':     f'P{current} ({difficulty})',
            'Difficulty': difficulty,
            'Time_s':     elapsed,
            'Move_type':  move[0]
        })
        result = game.make_move(*move)
        if result != 0 or not game.get_valid_moves():
            if result == -1 or result == 0:
                winner = 'Draw'
            else:
                winner = dif_p1 if result == 1 else dif_p2
            return winner, turn_data


def run_tournament(matchups, games_per_matchup=5):
    all_turns = []
    wins      = defaultdict(int)
    print("Starting MCTS tournament...")
    for dif1, dif2 in matchups:
        print(f"\n{dif1.upper()} vs {dif2.upper()} ({games_per_matchup} games):")
        for i in range(games_per_matchup):
            winner, turns = simulate_game(dif1, dif2)
            all_turns.extend(turns)
            wins[winner] += 1
            print(f"  Game {i+1}: winner = {winner}")
    return pd.DataFrame(all_turns), dict(wins)

In [ ]:
MATCHUPS          = [('facil', 'medio'), ('medio', 'dificil')]
GAMES_PER_MATCHUP = 5  # increase for more robust statistics

df_turns, win_counts = run_tournament(MATCHUPS, games_per_matchup=GAMES_PER_MATCHUP)
print("\nWin totals:", win_counts)

## 3.2 Results Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('MCTS Performance Analysis – PopOut Tournament', fontsize=15, fontweight='bold')

# Chart 1: Win counts
labels = list(win_counts.keys())
values = list(win_counts.values())
colors = {'facil': '#ff9999', 'medio': '#66b3ff', 'dificil': '#99ff99', 'Draw': '#c2c2f0'}
bar_colors = [colors.get(l, '#d3d3d3') for l in labels]
axes[0].bar(labels, values, color=bar_colors, edgecolor='white')
axes[0].set_title('Tournament Win Counts')
axes[0].set_ylabel('Number of Wins')
for i, v in enumerate(values):
    axes[0].text(i, v + 0.05, str(v), ha='center', fontweight='bold')

# Chart 2: Computational cost per difficulty (boxplot)
dif_order = [d for d in ['facil', 'medio', 'dificil'] if d in df_turns['Difficulty'].unique()]
sns.boxplot(ax=axes[1], x='Difficulty', y='Time_s', data=df_turns, order=dif_order)
axes[1].set_title('Computation Time per Move')
axes[1].set_ylabel('Seconds per Move')

# Chart 3: Drop vs Pop distribution per difficulty
move_dist = df_turns.groupby(['Difficulty', 'Move_type']).size().unstack(fill_value=0)
move_pct  = move_dist.div(move_dist.sum(axis=1), axis=0) * 100
move_pct  = move_pct.reindex([d for d in ['facil', 'medio', 'dificil'] if d in move_pct.index])
move_pct.plot(kind='bar', stacked=True, ax=axes[2], color=['#4daf4a', '#e41a1c'], edgecolor='white')
axes[2].set_title('Move Type Distribution (Drop vs Pop)')
axes[2].set_ylabel('Percentage (%)')
axes[2].legend(title='Move type')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('mcts_analysis.png', dpi=150)
plt.show()
print("Chart saved as 'mcts_analysis.png'.")

## 3.3 Discussion

The charts above highlight several key behaviours of our MCTS implementation:

**Win rates:**  
Higher difficulty levels consistently win more games. Hard beats Medium, which beats Easy. The gap between Easy and Medium is large because Easy uses a purely random simulation policy and only 150 iterations, while Medium uses 1 000 iterations with light heuristics.

**Computation time:**  
Easy moves are near-instantaneous (< 0.1 s). Medium moves average around 0.3–0.5 s. Hard moves are bounded by the 4.5-second time limit, meaning the agent genuinely exhausts its budget on complex positions.

**Pop usage:**  
Harder agents make slightly more `pop` moves relative to Easy, reflecting that the heuristic simulation policies (especially `heavy`) assign lower weights to pop moves, but the deeper search tree finds situations where popping is tactically decisive.

---

# Part 4 – Full Game Runner

The `play_game` function ties all components together and supports four modes:
1. **Human vs Human**
2. **Human vs MCTS** (configurable difficulty)
3. **MCTS vs MCTS** (both sides configurable)
4. **MCTS vs ID3 Tree** (trained above)

The human player selects moves by index from the printed list of legal moves.

In [ ]:
def get_human_move(game):
    moves = game.get_valid_moves()
    while True:
        try:
            print("\nLegal moves: ", end="")
            for i, (m_type, col) in enumerate(moves):
                print(f"[{i}] {m_type} {col} ", end=" ")
            idx = int(input("\nChoose move index: "))
            if 0 <= idx < len(moves):
                return moves[idx]
            print("Index out of range.")
        except ValueError:
            print("Invalid input. Enter a number.")


def play_game(p1_type, p2_type, tree=None, dif_p1='dificil', dif_p2='dificil'):
    game = PopOut()
    print("\n" + "="*50)
    print(f" GAME: {p1_type.upper()} vs {p2_type.upper()}")
    print("="*50)
    while True:
        game.print_board()
        if game.is_repetition_draw():
            print("\nGame ended in a DRAW (threefold repetition).")
            break
        is_p1       = (game.current_player == 1)
        tipo_atual  = p1_type if is_p1 else p2_type
        dif_atual   = dif_p1  if is_p1 else dif_p2
        if tipo_atual == 'humano':
            move = get_human_move(game)
        elif tipo_atual == 'mcts':
            print(f"\nPlayer {game.current_player} (MCTS – {dif_atual}) thinking...")
            move = get_mcts_move(game, dificuldade=dif_atual)
            print(f"  → Chose: {move}")
        elif tipo_atual == 'id3':
            print(f"\nPlayer {game.current_player} (ID3 Tree) thinking...")
            move = get_id3_move(game, tree)
            print(f"  → Chose: {move}")
        res = game.make_move(*move)
        if res != 0 or not game.get_valid_moves():
            game.print_board()
            if res in (-1, 0):
                print("\nGame ended in a DRAW.")
            else:
                print(f"\n*** WINNER: Player {res} ({'X' if res == 1 else 'O'}) ***")
            break

### CPU vs CPU Demo (MCTS Medium vs ID3 Tree)

In [ ]:
# Uncomment to run an autonomous game:
# play_game('mcts', 'id3', tree=tree_popout, dif_p1='medio')

# Uncomment to play as human vs MCTS:
play_game('humano', 'mcts', dif_p2='dificil')

print("To play, uncomment one of the lines above and re-run this cell.")


 GAME: HUMANO vs MCTS

  0   1   2   3   4   5   6
 ┌───┬───┬───┬───┬───┬───┬───┐
 │ . │ . │ . │ . │ . │ . │ . │
 ├───┼───┼───┼───┼───┼───┼───┤
 │ . │ . │ . │ . │ . │ . │ . │
 ├───┼───┼───┼───┼───┼───┼───┤
 │ . │ . │ . │ . │ . │ . │ . │
 ├───┼───┼───┼───┼───┼───┼───┤
 │ . │ . │ . │ . │ . │ . │ . │
 ├───┼───┼───┼───┼───┼───┼───┤
 │ . │ . │ . │ . │ . │ . │ . │
 ├───┼───┼───┼───┼───┼───┼───┤
 │ . │ . │ . │ . │ . │ . │ . │
 └───┴───┴───┴───┴───┴───┴───┘
 Turn: Player X

Legal moves: [0] drop 0  [1] drop 1  [2] drop 2  [3] drop 3  [4] drop 4  [5] drop 5  [6] drop 6  Invalid input. Enter a number.

Legal moves: [0] drop 0  [1] drop 1  [2] drop 2  [3] drop 3  [4] drop 4  [5] drop 5  [6] drop 6  
  0   1   2   3   4   5   6
 ┌───┬───┬───┬───┬───┬───┬───┐
 │ . │ . │ . │ . │ . │ . │ . │
 ├───┼───┼───┼───┼───┼───┼───┤
 │ . │ . │ . │ . │ . │ . │ . │
 ├───┼───┼───┼───┼───┼───┼───┤
 │ . │ . │ . │ . │ . │ . │ . │
 ├───┼───┼───┼───┼───┼───┼───┤
 │ . │ . │ . │ . │ . │ . │ . │
 ├───┼───┼───┼───┼───┼───

---

# Conclusions

This project explored two complementary AI paradigms applied to the game of **PopOut**:

### MCTS Agent
- The MCTS agent, enhanced with **UCT**, **heuristic simulation policies**, and a **discount factor** in backpropagation, provides a strong and scalable player.
- **Tactical vision** (1-step lookahead before tree search) eliminates obvious blunders at negligible cost.
- The difficulty system produces measurably different skill levels, confirmed by the tournament results.

### ID3 Decision Tree
- The ID3 tree trained on MCTS self-play data achieves 20–40% move accuracy — meaningful for a game with up to 14 legal moves per turn.
- On the **Iris dataset**, the same implementation achieves high accuracy (typically > 90%), demonstrating that the algorithm is correct and generalises well when the data is well-structured.
- **Discretisation** is the key enabler for applying ID3 to continuous-valued domains.

### Comparison

| | MCTS | ID3 |
|---|---|---|
| Knowledge source | Self-play search | Training data |
| Requires dataset | No | Yes |
| Interpretable | No | Yes |
| Skill ceiling | High (scales with compute) | Limited by data quality |
| Speed at inference | Slow (runs simulations) | Very fast (tree traversal) |

---

*End of notebook.*